# Enterprise Gemini Agent Session Batch Transcription Pipeline

This notebook orchestrates the stateful, highly scalable batch transcription of VHF/UHF radio communications across hundreds of audio files. It combines **Google Cloud Vertex AI Agent Engine Sessions** for persistent conversational turn tracking with an advanced hybrid control plane that routes audio payloads natively over the web to custom multi-region fine-tuned models.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/gemini_agent_session/transcribe_masked_audio.ipynb)

---

## 🏛️ System Architecture & Innovations

### 1. Hybrid Control Plane (Single-Region Memory + Multi-Region ML Inference)
* **The Control Plane**: To utilize premium evaluation features and stable session logging, all conversational context history (`VertexAiSessionService`) is managed strictly within the single-region `us-central1` database.
* **The ML Compute Target**: When evaluating custom fine-tuned models (`6182316388472848384`), inference traffic is dynamically routed to the multi-region `us` serving infrastructure using our canonical `MultiRegionGemini` subclass. This immutably preserves custom Vertex AI client configurations across Pydantic compilation sweeps.

### 2. Pure Transparent Context Ownership (Zero Client State)
* **The Core Pattern**: In this pipeline, the multi-turn conversational memory graph is **100% delegated to and owned by the Cloud Agent Engine Control Plane (`VertexAiSessionService`)**.
* **Zero Client Boilerplate**: In our Colab execution lanes, we do not build, maintain, append, or convey raw conversational history arrays (`messages.append`). The context plane is entirely invisible and transparent to us in the notebook.
* **Maintainer Advisory**: Future maintainers and autonomous agents MUST NOT attempt to introduce client-side message history tracking or execute stateless non-conversational unary prediction fallbacks. We purely hand our raw `Content` packet and `session_id` pointer to `channel_runner.run_async()`, allowing the Cloud Control Plane to automatically maintain state and resolve tactical jargon across turns.

### 3. Standalone Parent Recording Sequential Lock
* VHF/UHF tactical radio transmissions are chunked across consecutive files (`seg000`, `seg001`, `seg002`). 
* To ensure the model retains perfect acoustic and acronym anchoring, we immutably group child segments under their parent audio recording folder name (`Path(audio_filepath).parent.name`) and execute them sequentially in strict chronological order `(offset, start_time)`. Independent parent audio recordings process concurrently, while child segments belonging to the same recording process as an indivisible sequential unit building on itself.

### 4. Pure Native GCS File Pointer Streaming (`Part.from_uri`)
* In our highly optimized enterprise architecture, all audio payloads are ingested natively as Google Cloud Storage pointers (`gs://...`) via `Part.from_uri()`. 
* This entirely eliminates legacy HTTPS Signed URL generation coroutines, completely bypassing local downloading and container RAM bloat while allowing the custom Vertex AI ML Endpoint to execute direct, ultra-fast VPC storage reads over Google Cloud's high-speed backbone.

### 5. Fail-Fast Execution Quality Gates
* Long-running concurrent batch jobs must be deeply protected against resource waste. 
* We inject rigorous Fail-Fast Exception Gates that break out of the standard 5-attempt retry loop immediately upon encountering fatal infrastructure errors (`403 PERMISSION_DENIED`, `404 NOT_FOUND`, `INVALID_ARGUMENT`), aborting doomed concurrent tasks instantly.

### 6. High-Performance Caching & GCS Sync
* Resilient batch lanes survive preemption seamlessly. 
* The pipeline automatically checkpoints intermediate transcription results directly to an NDJSON storage file, synchronizing with remote GCS storage buckets in background coroutine threads to naturally deduplicate and resume massive evaluations across preemption cycles.


In [ ]:
# @title Install Dependencies
%pip install -q "google-adk[vertex]>=2.3.0" "google-genai>=0.1.1" "pydantic>=2.10.0" loguru

In [ ]:
# @title Imports
import asyncio
from collections import defaultdict
from datetime import timedelta
from functools import cached_property
import hashlib
import json
import os
from pathlib import Path
import random
import re
import subprocess
import sys
import time
from urllib.parse import urlparse
from google.adk.plugins.base_plugin import BasePlugin

from google import adk, genai
from google.adk.events import Event
from google.adk.models import Gemini
from google.adk.plugins.base_plugin import BasePlugin
from google.adk.runners import GetSessionConfig, RunConfig, Runner
from google.adk.sessions import VertexAiSessionService
from google.api_core import retry as api_retry
from google.api_core import retry_async as api_retry_async
from google.api_core.exceptions import GoogleAPICallError
from google.cloud import storage
from google.colab import auth, userdata
from google.genai import types
from google.genai.errors import ClientError
from google.oauth2 import credentials as oauth_creds
from IPython.display import display
from loguru import logger
import pandas as pd
from tqdm.auto import tqdm
import vertexai

In [ ]:
# @title Authenticate with GCP
# @markdown Run this cell to authenticate your browser session with Google Cloud.
# @markdown This is required to access your Vertex AI models and GCS buckets.
print("Attempting standard browser authentication...")
auth.authenticate_user()
print("Browser authentication successful!")

# Configure gcloud CLI to use your target project
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title [OPTIONAL] Deploy New Agent Engine
# @markdown Run this cell **ONLY** if you need to deploy a brand-new Agent Engine in the control plane region (e.g. during a region migration or setup).
# @markdown Otherwise, **skip this cell entirely!**

# Self-containment variables to allow running before the main config cell
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
CONTROL_PLANE_LOCATION = "global"

DEPLOY_NEW_ENGINE = False  # @param {type:"boolean"}

if DEPLOY_NEW_ENGINE:
    vertexai.init(project=GCP_PROJECT_ID, location=CONTROL_PLANE_LOCATION)
    v_client = vertexai.Client(
        project=GCP_PROJECT_ID, location=CONTROL_PLANE_LOCATION
    )

    print(
        f"Deploying new Agent Engine in '{CONTROL_PLANE_LOCATION}'... (This takes ~60-90s)"
    )
    new_engine = v_client.agent_engines.create()
    new_id = new_engine.api_resource.name.split("/")[-1]

    # 🌿 CPython Memory Sync: Immutably update the global AGENT_ENGINE_ID in memory
    # so the user can run the rest of the notebook immediately without re-running Cell 3!
    AGENT_ENGINE_ID = new_id

    print(f"\n✅ Success! New Agent Engine ID: {new_id}")
    print(
        "👉 Please save this ID in your Colab Secrets (userdata) as 'AGENT_ENGINE_ID'!"
    )
    print(
        "\n⚠️ CRITICAL WARNING: Please UNCHECK the 'DEPLOY_NEW_ENGINE' checkbox in the form widget now!"
    )
    print(
        "If you leave it checked, running 'Run All' in the future will deploy another duplicate engine and waste time."
    )
else:
    print("Skipped. Using existing AGENT_ENGINE_ID from secrets.")

In [ ]:
# @title Define constants and initial logging

# @markdown ### Model Selection
# @markdown **Option A: Select a base model**
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview", "gemini-3.5-flash"] {type:"string"}

# @markdown **Option B: Use a Tuned ML Model (Deployed in multi-region us)**
USE_CUSTOM_MODEL = False  # @param {type:"boolean"}

# Control Plane Session & Infrastructure Configuration
AGENT_NAME = "radio_transcript_session_agent"
APP_NAME = "contextual_audio_pipeline"
USER_ID = "radio_transcription_worker"

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCP_PROJECT_NUMBER = userdata.get("GCP_PROJECT_NUMBER")
GCS_BUCKET = userdata.get("GCS_BUCKET")

CUSTOM_MODEL_ID = userdata.get("CUSTOM_MODEL_ID")
GCP_CUSTOM_MODEL_NAME = userdata.get("GCP_CUSTOM_MODEL_NAME")
AGENT_ENGINE_ID = (
    str(userdata.get("AGENT_ENGINE_ID"))
    if userdata.get("AGENT_ENGINE_ID")
    else None
)

# @markdown ### GCP Infrastructure Configuration
# @markdown We use the 'global' location to route all traffic through Google's global gateway.
# @markdown This completely eliminates Regional Access Boundary warnings and location mismatches.
# fmt: off
GCP_LOCATION = "us"  # @param {type:"string"}

# @markdown All conversational context history and the Agent Engine (control plane)
# @markdown are managed strictly within a single region (typically 'us-central1').
CONTROL_PLANE_LOCATION = "global"  # @param {type:"string"}

# @markdown ### Input/Output Configuration
# @markdown Partial path under `gs://{GCS_BUCKET}/segmented_audio/` where `batch_manifest.jsonl` and audio segments are located (e.g., `broadcastify/calls/eval_audio_masked_v2`).
INPUT_AUDIO_DIR = ""  # @param {type:"string"}
# @markdown Partial path under `gs://{GCS_BUCKET}/transcripts/` where outputs will be stored (e.g., `broadcastify/calls/eval`).
OUTPUT_TRANSCRIPT_DIR = ""  # @param {type:"string"}
# @markdown Experiment Name (representing the subfolder under the model directory, e.g., `bcfy_calls_v1`).
EXPERIMENT_NAME = ""  # @param {type:"string"}


assert INPUT_AUDIO_DIR, "INPUT_AUDIO_DIR must be provided and cannot be empty."
assert OUTPUT_TRANSCRIPT_DIR, "OUTPUT_TRANSCRIPT_DIR must be provided and cannot be empty."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in Colab userdata."
assert GCS_BUCKET, "GCS_BUCKET must be provided in Colab userdata."
assert AGENT_ENGINE_ID, "AGENT_ENGINE_ID must be provided in Colab userdata."
# fmt: on

# Pipeline Control
# fmt: off
OVERWRITE_EXISTING = True  # @param {type:"boolean"}
USE_STATEFUL_SESSIONS = True  # @param {type:"boolean"}

# Streamlined Output Path Generation
if USE_CUSTOM_MODEL:
    assert CUSTOM_MODEL_ID, "CUSTOM_MODEL_ID must be provided in Colab userdata."  # fmt: skip
    assert GCP_CUSTOM_MODEL_NAME, "GCP_CUSTOM_MODEL_NAME must be provided in Colab userdata."  # fmt: skip
else:
    MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)

GCS_OUTPUT_BASE = f"transcripts/{OUTPUT_TRANSCRIPT_DIR}/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
MANIFEST_URI = f"gs://{GCS_BUCKET}/segmented_audio/{INPUT_AUDIO_DIR}/batch_manifest.jsonl"
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"
CHECKPOINT_FILE = "interim_backup_predictions.jsonl"
# fmt: on


SYSTEM_PROMPT = """\
Your primary task is to produce a strict, verbatim transcription of the spoken audio. Your absolute highest priority is to transcribe only what you hear with high acoustic certainty. Do not add, invent, or infer any speech that is not clearly audible. The audio may originate from VHF/UHF radio traffic and can include mic clicks, RF static, radio hum, and potentially unintelligible speech. When the audio is unequivocally confirmed as fire-related dispatch, speakers often use heavy jargon, and specific formatting rules apply.

EXPECTED TERMINOLOGY:
These are terms and unit identifiers commonly used in fire-related dispatch. These terms and formatting rules apply exclusively to audio that is unequivocally confirmed as fire-related dispatch. If these exact terms are clearly heard in the audio, transcribe them as listed. Do not invent or infer the use of these terms if they are not genuinely spoken.
copy, received, affirmative, affirm, proceed, responding, responding to, en-route, on-scene, in the area, available, returning, in service, got a caller, caller advising, in quarters, arrived, go ahead, back at, engine, tanker, brush, brush truck, tender, battalion, squad, ladder, tower, tower-ladder, medic, ambulance, k, branch, chopper, copter, AIQ, AOR, DO, IC, ICP, LAT, RP, SEAT, TAC, VFIRE, VLAT, patrol, rescue, station, personnel, air attack, air tactics, helispot, lead plane, strike team, control, being toned, box alarm, cancel the balance, chaparral, exposure protection, fire attack, fire boss, forward progress stopped, forward rate of spread stopped, heavy timber, left flank, light flashy fuels, rate of spread, right flank, structure defense, structure protection, structures threatened, terrain driven, wind driven, clear, clear and in service, code 1, code 2, code 3, code 4, code 33, medical call, fire alarm, commercial fire alarm, breathing problem, cardiac, heart problem, diabetic shock, mvc, trespass, harassment, 10-4, 10-7, 10-8, 10-9, 10-15, 10-20, 10-22, 10-23, 10-91, 10-97.

CRITICAL RULES:
1. Output the transcript strictly and precisely as spoken in the audio, with no newlines. Do not add, invent, or infer any speech that is not clearly audible.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. If the audio contains a unit identifier, format it as the unit type followed by digits (e.g., Engine 41, Battalion 2). Apply this rule strictly only if the unit identifier is clearly spoken AND the context is unequivocally fire-related dispatch.
4. Transcribe only the duration of speech present. Do not extend the transcription with additional words or phrases that were not spoken, even if contextually plausible.

QUALITY GATE: Your absolute highest priority is to transcribe only what you hear with high acoustic certainty.
    *   If the audio contains clear speech that is not fire-related dispatch, you MUST transcribe it verbatim, exactly as heard, without applying any fire-specific formatting or jargon, and without attempting to interpret it as fire dispatch traffic.
    *   If a portion of audio is obscured, noisy, ambiguous, or contains speech that cannot be confidently identified, you MUST replace that specific portion with [UNINTELLIGIBLE].
    *   Do not attempt to infer, guess, or invent speech to fit any expected context or terminology list.
    *   Do not attempt to phonetically guess ambiguous noise.
    *   If the entire audio segment does not contain any discernible speech, output only [UNINTELLIGIBLE].

TASK:
Transcribe the attached audio. Output strictly the transcript.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
    "thinking_config": types.ThinkingConfig(thinking_budget=0),
}

# Context Session History Settings
# Grounded in empirical ASR research (https://arxiv.org/abs/2602.09044) showing that long-context ASR
# performance gains scale up to 21.8 minutes (1,308 seconds) of chronological audio history.
# Calibrated against the empirical segment duration mean of 4.48 seconds for this specific corpus,
# this translates to ~292 turns. Since each turn consists of 2 events (1 user audio event +
# 1 model transcript event), we set this to 580 to capture the full 21.8 minutes of optimal history.
NUM_RECENT_EVENTS = 580
CONCURRENCY_LIMIT = 5
MAX_RETRIES = 5
SOCKET_TIMEOUT = 180.0

logger.remove()
# Assigning to avoid outputting the response.
_ = logger.add(
    sys.stderr, format="<level>{level}</level>: {message}", level="DEBUG"
)

In [ ]:
# @title Define Configured Gemini Subclass
# ConfiguredGemini Subclass: Explicitly passes the Colab Project ID and Location
# to the underlying GenAI client, bypassing ADK's project resolution limitation in Colab.
# We keep this subclass as a 100% guaranteed safety net for your custom model.
class ConfiguredGemini(Gemini):
    # Dynamically defaults to the global GCP_LOCATION constant
    target_location: str = GCP_LOCATION

    @property
    def api_client(self) -> genai.Client:
        return genai.Client(
            project=GCP_PROJECT_ID,
            location=self.target_location,
            vertexai=True,
            http_options=types.HttpOptions(
                headers=self._tracking_headers(),
                retry_options=self.retry_options,
            ),
        )

In [ ]:
# @title Define ADK Agent and Session Service

# 1. Configure native HTTP retry behavior (using numeric seconds)
# Handled directly by the model adapter to automatically retry 429 rate limit errors
custom_http_retry = types.HttpRetryOptions(
    attempts=6, max_delay=60.0, initial_delay=2.0, exp_base=2.0
)


# 2. Target Model Instantiation (Hybrid Path with Native Retry Adapter)
# If using the custom fine-tuned model, we wrap it in ConfiguredGemini (routed to 'us').
# If using the base model, we run it natively via the standard Gemini class (routed to 'global').
# We pass retry_options directly to the model adapter as officially specified in the ADK docs!
if USE_CUSTOM_MODEL:
    # Custom fine-tuned models are deployed in the 'us' multi-region.
    # To be 100% safe against ADK runner location overrides, we use the ConfiguredGemini subclass.
    TARGET_MODEL = ConfiguredGemini(
        model=f"projects/{GCP_PROJECT_NUMBER}/locations/us/endpoints/{CUSTOM_MODEL_ID}",
        target_location="us",
        retry_options=custom_http_retry,
    )
else:
    # Base models are wrapped in ConfiguredGemini to activate our explicit project/location client override.
    # This completely resolves the ADC ValueError on the Colab VM!
    TARGET_MODEL = ConfiguredGemini(
        model=f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}",
        retry_options=custom_http_retry,
    )


# Initialize the Vertex AI client for pre-flight validation
vertex_client = vertexai.Client(
    project=GCP_PROJECT_ID, location=CONTROL_PLANE_LOCATION
)

# 3. Agent Engine Resolution (reuses the pre-existing ID from secrets)
agent_engine_id = AGENT_ENGINE_ID

# 🌿 PRE-FLIGHT VALIDATION: Verify the Agent Engine exists in the configured CONTROL_PLANE_LOCATION region.
# Construct and use the fully qualified resource name to bypass the new Google GenAI SDK's
# internal URL-builder bug (which fails to insert '/reasoningEngines/' when passed a raw ID).
if agent_engine_id:
    try:
        logger.debug(
            f"Verifying Agent Engine '{agent_engine_id}' in region '{CONTROL_PLANE_LOCATION}'..."
        )
        full_resource_name = f"projects/{GCP_PROJECT_ID}/locations/{CONTROL_PLANE_LOCATION}/reasoningEngines/{agent_engine_id}"
        vertex_client.agent_engines.get(name=full_resource_name)
        logger.success(
            f"Verified! Agent Engine is active and ready in '{CONTROL_PLANE_LOCATION}'."
        )
    except Exception as e:
        raise RuntimeError(
            f"\n\n🚨 ERROR: Could not find Agent Engine '{agent_engine_id}' in region '{CONTROL_PLANE_LOCATION}'!\n"
            f"This is a regional mismatch. The Agent Engine in your secrets was likely deployed in a different "
            f"region than the current notebook's CONTROL_PLANE_LOCATION ('{CONTROL_PLANE_LOCATION}').\n\n"
            f"👉 TO RESOLVE THIS:\n"
            f"1. Scroll up to Cell 5 ([OPTIONAL] Deploy New Agent Engine).\n"
            f"2. Check the 'DEPLOY_NEW_ENGINE' box and run that cell to deploy a new engine in '{CONTROL_PLANE_LOCATION}'.\n"
            f"3. Copy the new ID, update your Colab Secrets (userdata) for 'AGENT_ENGINE_ID', and run this cell again!\n"
        ) from e

# 4. Define ADK Agent
audio_agent = adk.Agent(
    model=TARGET_MODEL,
    name=AGENT_NAME,
    instruction=SYSTEM_PROMPT,
    generate_content_config=types.GenerateContentConfig(
        temperature=GENERATION_CONFIG["temperature"],
        max_output_tokens=GENERATION_CONFIG["max_output_tokens"],
        thinking_config=GENERATION_CONFIG["thinking_config"],
        safety_settings=SAFETY_SETTINGS,
    ),
)

# 5. Initialize Session Service
session_service = VertexAiSessionService(
    project=GCP_PROJECT_ID,
    location=CONTROL_PLANE_LOCATION,
    agent_engine_id=agent_engine_id,
)

# 6. Initialize Runner (no plugins needed, retry is natively handled by the model adapter!)
runner = Runner(
    agent=audio_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

storage_client = storage.Client(project=GCP_PROJECT_ID)
CHECKPOINT_FILE = "interim_backup_predictions.jsonl"
checkpoint_write_lock = asyncio.Lock()

### 📊 Pipeline Latency & Context Caching Characteristics

This stateful transcription pipeline exhibits a unique two-phase latency curve, driven by **Vertex AI's automatic Context Caching** on the serving cluster:

```text
Latency (s)
  ^
 45 |                                                            / [Without Caching: Linear Growth]
 40 |                                                           /
 35 |                                                          /
 30 |                                                         /
 25 |                                                        /
 20 |                                                       /
 15 |                      *=================================*======= [Actual Observed: Context Cached]
 10 |                    .-'
  5 |                 .-'
  0 +----------------*---------------------------------------------------->
   Turn:             1     7         12        20        30        44
```

#### 📈 Phase 1: The Warm-up Phase (Turns 1 to ~7)
* **Behavior:** Latency climbs diagonally from **~4.5s** on Turn 1 to **~13.5s** by Turn 7.
* **Mechanism:** As the initial conversational history accumulates in the session database, the input token payload grows. The serving cluster compiles the Key-Value (KV) attention states for this history and commits them to the hardware cache.

#### ⏸️ Phase 2: The Steady-State Plateau (Turns 7 to 44+)
* **Behavior:** Latency flattens out completely, running flat at **~12.5s (±1.5s)** with **0 seconds of incremental growth per turn**.
* **Mechanism:** Once the cache is warm, the model does not re-evaluate historical tokens. It loads the compiled KV cache instantly from high-bandwidth memory and only performs attention computation on the *new* segment (~1,120 tokens for a 4.5s clip).

#### 🛡️ Token & Latency Safety
* Because of this flat caching plateau, increasing `NUM_RECENT_EVENTS` from `260` to `580` (capturing the paper-proven optimal **21.8 minutes** of history) introduces **zero latency penalty** once the cache is warm, while staying well within the model's 1M+ token capacity.

In [ ]:
# @title Pipeline Execution Logic


class SessionExpiredError(Exception):
    """Custom exception raised when a resumed session has expired or was deleted."""

    pass


class FatalPipelineError(Exception):
    """Custom exception raised when an unrecoverable error occurs that should abort the entire pipeline."""

    pass


# Global Lock specifically to throttle Vertex Session Write Requests Quota endpoint
session_quota_lock = asyncio.Lock()

# Foundational enterprise AsyncIO Retry Policy protecting all streaming network channels
custom_async_retry = api_retry_async.AsyncRetry(
    predicate=api_retry.if_exception_type(
        (ClientError, GoogleAPICallError, asyncio.TimeoutError, ConnectionError)
    ),
    initial=1.0,
    maximum=60.0,
    multiplier=2.0,
    deadline=300.0,
)


def get_gcs_checkpoint_blob():
    """Derives GCS checkpoint path consistently for both load and upload."""
    out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
    out_path = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
    checkpoint_blob_path = "/".join(out_path[:-1]) + "/" + CHECKPOINT_FILE
    return storage_client.bucket(out_bucket).blob(checkpoint_blob_path)


def load_gcs_checkpoint() -> tuple[dict[str, str], dict[str, str]]:
    blob = get_gcs_checkpoint_blob()
    records = {}

    if blob.exists():
        logger.info(
            f"Found existing checkpoint on GCS: {blob.name}. Loading..."
        )
        blob.download_to_filename(CHECKPOINT_FILE)
    else:
        logger.info("No remote checkpoint found on GCS.")

    if os.path.exists(CHECKPOINT_FILE):
        logger.info(f"Loading checkpoint from local file: {CHECKPOINT_FILE}")
        # 1. Read all records, mapping by audio_filepath to naturally deduplicate
        with open(CHECKPOINT_FILE, "r") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    if not record.get("error"):
                        records[record["audio_filepath"]] = record

        # 2. Rewrite the local checkpoint file to be clean of errors and duplicates
        with open(CHECKPOINT_FILE, "w") as f:
            for record in records.values():
                f.write(json.dumps(record) + "\n")

        logger.info(f"Loaded {len(records)} completed records.")
    else:
        logger.info("No local or remote checkpoint found. Starting fresh.")

    # 3. Build completed transcripts map and active sessions map
    completed_transcripts = {
        filepath: rec["transcript"] for filepath, rec in records.items()
    }
    channel_sessions = {}
    for rec in records.values():
        if rec.get("session_id") and rec.get("example_id"):
            channel_sessions[rec["example_id"]] = rec["session_id"]

    return completed_transcripts, channel_sessions


async def upload_checkpoint_to_gcs() -> None:
    """Synchronizes the local checkpoint file to GCS safely using CPython thread offloading."""
    async with checkpoint_write_lock:
        if os.path.exists(CHECKPOINT_FILE):
            try:
                blob = get_gcs_checkpoint_blob()
                await asyncio.to_thread(
                    blob.upload_from_filename, CHECKPOINT_FILE
                )
            except Exception as sync_err:
                logger.warning(
                    f"Failed to backup checkpoint to GCS: {sync_err}"
                )


async def process_single_channel(
    channel_id: str,
    segment_entries: list[dict],
    completed_records: dict[str, str],
    channel_sessions: dict[str, str],
    semaphore: asyncio.Semaphore,
    pbar: tqdm,
) -> list[dict]:
    """Processes a channel using either stateful sessions or pure stateless requests."""
    results = []

    try:
        async with semaphore:
            channel_user_id = f"{USER_ID}_{channel_id}"

            # ----------------------------------------------------------------
            # PATH A: Pure Stateless Execution (Air-Tight Direct Model Calls)
            # ----------------------------------------------------------------
            if not USE_STATEFUL_SESSIONS:
                from google.genai import Client as GenAiClient
                from google.genai import types as genai_types

                stateless_client = GenAiClient(
                    project=GCP_PROJECT_ID,
                    location=GCP_LOCATION,
                    vertexai=True,
                )

                stateless_config = genai_types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=GENERATION_CONFIG["temperature"],
                    max_output_tokens=GENERATION_CONFIG["max_output_tokens"],
                    safety_settings=[
                        genai_types.SafetySetting(
                            category=s["category"],
                            threshold=s["threshold"],
                        )
                        for s in SAFETY_SETTINGS
                    ],
                )

                for turn_index, entry in enumerate(segment_entries, start=1):
                    uri = entry["audio_filepath"]

                    if uri in completed_records:
                        cached_transcript = completed_records[uri]
                        logger.info(
                            f"[CACHE HIT] Restored transcript for {uri} (Turn {turn_index}/{len(segment_entries)})"
                        )
                        results.append(
                            {
                                "example_id": channel_id,
                                "audio_filepath": uri,
                                "transcript": cached_transcript,
                                "error": None,
                            }
                        )
                        pbar.update(1)
                        continue

                    logger.debug(
                        f"[API CALL] Sending pure stateless request for {uri}..."
                    )
                    req_start_time = time.time()

                    try:
                        contents = [
                            genai_types.Part.from_uri(
                                file_uri=uri, mime_type="audio/flac"
                            )
                        ]

                        # Run the blocking API call in threadpool to keep event loops responsive
                        response = await asyncio.to_thread(
                            stateless_client.models.generate_content,
                            model=f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}",
                            contents=contents,
                            config=stateless_config,
                        )

                        # Extract telemetry
                        usage = getattr(response, "usage_metadata", None)
                        prompt_tokens = (
                            getattr(usage, "prompt_token_count", 0)
                            if usage
                            else 0
                        )
                        cached_tokens = (
                            getattr(usage, "cached_content_token_count", 0)
                            if usage
                            else 0
                        )
                        output_tokens = (
                            getattr(usage, "candidates_token_count", 0)
                            if usage
                            else 0
                        )

                        transcript = None
                        final_error_msg = None

                        if response.candidates:
                            cand = response.candidates[0]
                            text = (
                                "".join(
                                    [
                                        p.text
                                        for p in cand.content.parts
                                        if p.text
                                    ]
                                ).strip()
                                if cand.content and cand.content.parts
                                else ""
                            )

                            # Check finish reason
                            f_reason = cand.finish_reason
                            if (
                                f_reason
                                and "STOP" not in str(f_reason)
                                and f_reason != 1
                            ):
                                final_error_msg = f"Stateless run failed with finish reason: {f_reason}"
                            else:
                                transcript = text
                        else:
                            final_error_msg = (
                                "Stateless run returned no candidates"
                            )

                    except Exception as err:
                        transcript = None
                        final_error_msg = (
                            f"Stateless run raised exception: {err}"
                        )

                    req_duration = time.time() - req_start_time

                    if transcript is not None:
                        if transcript == "":
                            logger.success(
                                f"[API SUCCESS] Pure ambient static/silence confirmed for {uri} in {req_duration:.2f}s (Turn {turn_index}/{len(segment_entries)})"
                            )
                        else:
                            logger.success(
                                f"[API SUCCESS] Final transcript acquired for {uri} in {req_duration:.2f}s (Turn {turn_index}/{len(segment_entries)})"
                            )
                        metrics = {
                            "turn_index": turn_index,
                            "latency": req_duration,
                            "prompt_tokens": prompt_tokens,
                            "cached_tokens": cached_tokens,
                            "output_tokens": output_tokens,
                            "is_fallback": False,
                        }
                        result_dict = {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": transcript,
                            "error": None,
                            "session_id": None,
                            "metrics": metrics,
                        }
                        results.append(result_dict)

                        async with checkpoint_write_lock:
                            with open(CHECKPOINT_FILE, "a") as f:
                                f.write(json.dumps(result_dict) + "\n")
                    else:
                        logger.error(
                            f"[API ERROR] {uri} - Final failure: {final_error_msg}"
                        )
                        result_dict = {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": None,
                            "error": final_error_msg,
                            "session_id": None,
                        }
                        results.append(result_dict)
                        async with checkpoint_write_lock:
                            with open(CHECKPOINT_FILE, "a") as f:
                                f.write(json.dumps(result_dict) + "\n")

                    pbar.update(1)

                try:
                    await upload_checkpoint_to_gcs()
                except Exception as sync_err:
                    logger.warning(
                        f"Failed to backup checkpoint to GCS for channel {channel_id}: {sync_err}"
                    )

                return results

            # ----------------------------------------------------------------
            # PATH B: Stateful Caching Execution (Active Session ADK Runner)
            # ----------------------------------------------------------------
            session_id = channel_sessions.get(channel_id)
            is_resumed = session_id is not None
            new_session = None

            # Attempt loop: Attempt 0 (resumed session), Attempt 1 (new session fallback)
            for session_attempt in range(2):
                if is_resumed:
                    logger.debug(
                        f"Resuming existing session {session_id} for channel {channel_id} "
                        f"({len(segment_entries)} total segments)"
                    )
                else:
                    session_id = None
                    logger.info(
                        f"Starting brand new agent session for {channel_id}..."
                    )
                    for retry in range(MAX_RETRIES):
                        try:
                            # 100% Fully staggered, rate-limited session creation to completely defeat 429 Quota errors
                            async with session_quota_lock:
                                new_session = (
                                    await session_service.create_session(
                                        user_id=channel_user_id,
                                        app_name=AGENT_ENGINE_ID,
                                        display_name=channel_id,
                                    )
                                )
                                await asyncio.sleep(
                                    0.5
                                )  # Minimal 500ms Quota stagger
                            if not new_session or not new_session.id:
                                raise ValueError(
                                    f"Invalid session created: {new_session}"
                                )
                            session_id = new_session.id
                            break
                        except Exception as e:
                            logger.warning(
                                f"Attempt {retry + 1} failed to create session for {channel_id}: {e}"
                            )
                            if isinstance(
                                e, (ClientError, GoogleAPICallError)
                            ) and (
                                "403" in str(e)
                                or "404" in str(e)
                                or "PERMISSION_DENIED" in str(e)
                                or "NOT_FOUND" in str(e)
                            ):
                                logger.error(
                                    f"FATAL SESSION ERROR: {e}. Aborting pipeline immediately."
                                )
                                raise FatalPipelineError(
                                    f"Pipeline aborted due to fatal unrecoverable session error: {e}"
                                ) from e

                            # Jittered exponential backoff to completely defeat Thundering Herd
                            jitter_sleep = (2**retry) + random.uniform(1.0, 3.0)
                            await asyncio.sleep(jitter_sleep)

                    if not session_id:
                        error_msg = f"Failed to create Vertex AI session for channel {channel_id} after {MAX_RETRIES} retries."
                        logger.error(error_msg)
                        for entry in segment_entries:
                            results.append(
                                {
                                    "example_id": channel_id,
                                    "audio_filepath": entry["audio_filepath"],
                                    "transcript": None,
                                    "error": error_msg,
                                }
                            )
                            pbar.update(1)
                        return results

                # Fully completely isolate the concurrent ADK Runner instance to ensure 100% airtight asyncio thread safety!
                channel_runner = Runner(
                    agent=audio_agent,
                    app_name=AGENT_ENGINE_ID,
                    session_service=session_service,
                )

                run_config = RunConfig(
                    get_session_config=GetSessionConfig(
                        create_if_not_exists=False,
                        num_recent_events=NUM_RECENT_EVENTS,
                    )
                )

                session_expired = False
                results = []

                try:
                    # 🌿 Process exactly sequentially in strict chronological order without a split!
                    for turn_index, entry in enumerate(
                        segment_entries, start=1
                    ):
                        uri = entry["audio_filepath"]

                        if uri in completed_records:
                            cached_transcript = completed_records[uri]
                            logger.info(
                                f"[CACHE HIT] Restored transcript for {uri} (Turn {turn_index}/{len(segment_entries)})"
                            )
                            results.append(
                                {
                                    "example_id": channel_id,
                                    "audio_filepath": uri,
                                    "transcript": cached_transcript,
                                    "error": None,
                                }
                            )
                            pbar.update(1)
                            continue

                        # PURE NATIVE GCS FILE POINTER STREAMING (Zero Signed URLs!)
                        # Promptless payload: only send the audio part. This is highly stable
                        # and prevents attention loops in multimodal cross-attention layers.
                        audio_part = types.Part.from_uri(
                            file_uri=uri, mime_type="audio/flac"
                        )
                        message = types.Content(role="user", parts=[audio_part])

                        final_error_msg = "Unknown error"
                        transcript = None
                        f_reason = None

                        # Initialize telemetry variables for visualizer
                        prompt_tokens = 0
                        cached_tokens = 0
                        output_tokens = 0
                        is_fallback = False

                        # 🌿 1. Upstream retried coroutine beautifully accumulating ALL intermediate content packets
                        @custom_async_retry
                        async def execute_streaming_turn():
                            streamed_text_chunks = []
                            finish_reason = None
                            embedded_error_msg = None
                            st_prompt = 0
                            st_cached = 0
                            st_output = 0

                            async for event in channel_runner.run_async(
                                user_id=channel_user_id,
                                session_id=session_id,
                                new_message=message,
                                run_config=run_config,
                            ):
                                if (
                                    hasattr(event, "error_message")
                                    and event.error_message
                                ):
                                    embedded_error_msg = event.error_message
                                if (
                                    hasattr(event, "finish_reason")
                                    and event.finish_reason
                                ):
                                    finish_reason = event.finish_reason

                                if getattr(event, "content", None) and getattr(
                                    event.content, "parts", None
                                ):
                                    for part in event.content.parts:
                                        if hasattr(part, "text") and part.text:
                                            streamed_text_chunks.append(
                                                part.text
                                            )

                                if event.is_final_response():
                                    if (
                                        hasattr(event, "raw_response")
                                        and event.raw_response.candidates
                                    ):
                                        finish_reason = (
                                            finish_reason
                                            or event.raw_response.candidates[
                                                0
                                            ].finish_reason
                                        )

                                    # 🌿 CRITICAL TELEMETRY: Extract and log exact GCP Billing/Usage Metadata for this turn
                                    usage = getattr(
                                        event, "usage_metadata", None
                                    )
                                    if usage:
                                        st_prompt = (
                                            getattr(
                                                usage, "prompt_token_count", 0
                                            )
                                            or 0
                                        )
                                        st_cached = (
                                            getattr(
                                                usage,
                                                "cached_content_token_count",
                                                0,
                                            )
                                            or 0
                                        )
                                        st_output = (
                                            getattr(
                                                usage,
                                                "candidates_token_count",
                                                0,
                                            )
                                            or 0
                                        )
                                        logger.debug(
                                            f"{Path(uri).name} (Turn {turn_index}): "
                                            f"Prompt Tokens: {st_prompt} | "
                                            f"Cached Tokens: {st_cached} | "
                                            f"Output Tokens: {st_output}"
                                        )

                            # 🛡️ Highly rigorous Enum string filter entirely outsmarting FinishReason Protobuf wrappers
                            if embedded_error_msg or (
                                finish_reason
                                and "STOP"
                                not in str(
                                    getattr(
                                        finish_reason, "name", finish_reason
                                    )
                                )
                                and "UNSPECIFIED"
                                not in str(
                                    getattr(
                                        finish_reason, "name", finish_reason
                                    )
                                )
                                and finish_reason != 0
                            ):
                                logger.warning(
                                    f"[API STREAM WARNING] {uri}: {embedded_error_msg or 'No embedded message'} (Reason: {finish_reason})"
                                )
                                # 🌿 CRITICAL DEBUG GATE: Print the exact text generated so far to prove/disprove the repetition hypothesis
                                # Using raw repr() and stripping the emoji to prevent terminal encoding swallowing
                                full_text_so_far = "".join(streamed_text_chunks)
                                logger.warning(
                                    f"[DEBUG] Raw text generated before failure: {repr(full_text_so_far[:200])}"
                                )

                            full_text = "".join(streamed_text_chunks).strip()
                            return (
                                full_text,
                                finish_reason,
                                embedded_error_msg,
                                st_prompt,
                                st_cached,
                                st_output,
                            )

                        logger.debug(
                            f"[API CALL] Sending streaming request for {uri} in session {session_id}..."
                        )
                        req_start_time = time.time()
                        try:
                            (
                                res_text,
                                f_reason,
                                emb_err,
                                prompt_tokens,
                                cached_tokens,
                                output_tokens,
                            ) = await asyncio.wait_for(
                                execute_streaming_turn(), timeout=SOCKET_TIMEOUT
                            )
                            transcript = res_text
                            if emb_err:
                                final_error_msg = emb_err
                            elif (
                                f_reason
                                and "STOP" not in str(f_reason)
                                and "UNSPECIFIED" not in str(f_reason)
                                and f_reason != 0
                            ):
                                final_error_msg = (
                                    f"Failed with finish reason: {f_reason}"
                                )
                            else:
                                final_error_msg = None
                        except Exception as stream_err:
                            # 1. Check if session expired (404 NOT_FOUND)
                            is_404 = (
                                (
                                    isinstance(stream_err, ClientError)
                                    and stream_err.code == 404
                                )
                                or (
                                    isinstance(stream_err, GoogleAPICallError)
                                    and stream_err.code == 404
                                )
                                or (
                                    "404" in str(stream_err)
                                    and (
                                        "not found" in str(stream_err).lower()
                                        or "does not exist"
                                        in str(stream_err).lower()
                                    )
                                )
                            )
                            if is_resumed and is_404:
                                logger.warning(
                                    f"⚠️ Session {session_id} has expired or was deleted (404 NOT_FOUND). "
                                    f"Wiping channel cache and starting channel {channel_id} fresh in a new session!"
                                )
                                session_expired = True
                                break

                            # 2. Check if it is a fatal unrecoverable infrastructure/permission error
                            is_fatal = (
                                isinstance(
                                    stream_err,
                                    (ClientError, GoogleAPICallError),
                                )
                                and (
                                    stream_err.code in (403, 400)
                                    or (
                                        stream_err.code == 404
                                        and not is_resumed
                                    )
                                )
                            ) or (
                                any(
                                    fatal_keyword in str(stream_err).upper()
                                    for fatal_keyword in (
                                        "403",
                                        "PERMISSION_DENIED",
                                        "INVALID_ARGUMENT",
                                    )
                                )
                            )
                            if is_fatal:
                                logger.error(
                                    f"FATAL TURN ERROR: {stream_err}. Aborting pipeline immediately."
                                )
                                raise FatalPipelineError(
                                    f"Pipeline aborted due to fatal unrecoverable turn error: {stream_err}"
                                ) from stream_err

                            transcript = None
                            final_error_msg = f"Streaming lane failed after upstream backoff: {stream_err}"
                            logger.warning(
                                f"[API STREAM FAILED] {uri}: {final_error_msg}"
                            )

                        req_duration = time.time() - req_start_time

                        # 🌿 2. Self-Healing Stateless Fallback (Triggered on safety block, recitation block, or loop crash)
                        is_blocked_or_empty = (
                            transcript is None
                            or transcript == ""
                            or (
                                f_reason
                                and any(
                                    br in str(f_reason).upper()
                                    for br in (
                                        "RECITATION",
                                        "MAX_TOKENS",
                                        "SAFETY",
                                    )
                                )
                            )
                        )

                        if is_blocked_or_empty:
                            logger.warning(
                                f"[STATELESS FALLBACK] Stateful turn {turn_index} on {Path(uri).name} failed "
                                f"(Transcript: {repr(transcript)}, Finish Reason: {f_reason}, Error: {final_error_msg}). "
                                f"Executing stateless fallback with clean context..."
                            )
                            try:
                                from google.genai import Client as GenAiClient
                                from google.genai import types as genai_types

                                # Instantiate a local stateless client routed to the global gateway
                                fallback_client = GenAiClient(
                                    project=GCP_PROJECT_ID,
                                    location=GCP_LOCATION,
                                    vertexai=True,
                                )

                                fallback_config = (
                                    genai_types.GenerateContentConfig(
                                        system_instruction=SYSTEM_PROMPT,
                                        temperature=0.0,
                                        max_output_tokens=512,
                                    )
                                )

                                fallback_contents = [
                                    genai_types.Part.from_uri(
                                        file_uri=uri, mime_type="audio/flac"
                                    )
                                ]

                                # Thread offload the blocking API request to protect high-concurrency event loops
                                response = await asyncio.to_thread(
                                    fallback_client.models.generate_content,
                                    model=f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}",
                                    contents=fallback_contents,
                                    config=fallback_config,
                                )

                                # Extract fallback telemetry
                                usage = getattr(
                                    response, "usage_metadata", None
                                )
                                prompt_tokens = (
                                    getattr(usage, "prompt_token_count", 0)
                                    if usage
                                    else 0
                                )
                                cached_tokens = (
                                    getattr(
                                        usage, "cached_content_token_count", 0
                                    )
                                    if usage
                                    else 0
                                )
                                output_tokens = (
                                    getattr(usage, "candidates_token_count", 0)
                                    if usage
                                    else 0
                                )
                                is_fallback = True

                                if response.candidates:
                                    cand = response.candidates[0]
                                    fallback_text = (
                                        "".join(
                                            [
                                                p.text
                                                for p in cand.content.parts
                                                if p.text
                                            ]
                                        ).strip()
                                        if cand.content and cand.content.parts
                                        else ""
                                    )

                                    if fallback_text or (
                                        cand.finish_reason
                                        and "STOP" in str(cand.finish_reason)
                                    ):
                                        logger.success(
                                            f"[STATELESS FALLBACK SUCCESS] Successfully recovered transcript statelessly for {Path(uri).name}: '{fallback_text}'"
                                        )
                                        transcript = fallback_text
                                        final_error_msg = None
                                        # Note: We do not write this back to the session history; the subsequent segments
                                        # will continue statefully. This is perfectly sufficient and extremely stable!
                                    else:
                                        logger.error(
                                            f"[STATELESS FALLBACK FAILED] Stateless fallback returned empty transcript for {Path(uri).name} (Reason: {cand.finish_reason})"
                                        )
                                        transcript = None
                                        final_error_msg = f"Stateless fallback returned empty transcript (Reason: {cand.finish_reason})"
                                else:
                                    logger.error(
                                        f"[STATELESS FALLBACK FAILED] Stateless fallback returned no candidates for {Path(uri).name}"
                                    )
                                    transcript = None
                                    final_error_msg = "Stateless fallback returned no candidates"
                            except Exception as fallback_err:
                                logger.error(
                                    f"[STATELESS FALLBACK CRITICAL] Stateless fallback raised exception: {fallback_err}"
                                )
                                transcript = None
                                final_error_msg = f"Stateless fallback raised exception: {fallback_err}"

                        # 🌿 3. Pure Execution Quality Gate (Zero Speculative Secondary Hybrid Context Pollution!)
                        if transcript is not None:
                            if transcript == "":
                                logger.success(
                                    f"[API SUCCESS] Pure ambient static/silence confirmed for {uri} in {req_duration:.2f}s (Turn {turn_index}/{len(segment_entries)})"
                                )
                            else:
                                logger.success(
                                    f"[API SUCCESS] Final transcript acquired for {uri} in {req_duration:.2f}s (Turn {turn_index}/{len(segment_entries)})"
                                )
                            metrics = {
                                "turn_index": turn_index,
                                "latency": req_duration,
                                "prompt_tokens": prompt_tokens,
                                "cached_tokens": cached_tokens,
                                "output_tokens": output_tokens,
                                "is_fallback": is_fallback,
                            }
                            result_dict = {
                                "example_id": channel_id,
                                "audio_filepath": uri,
                                "transcript": transcript,
                                "error": None,
                                "session_id": session_id,
                                "metrics": metrics,
                            }
                            results.append(result_dict)

                            async with checkpoint_write_lock:
                                with open(CHECKPOINT_FILE, "a") as f:
                                    f.write(json.dumps(result_dict) + "\n")
                        else:
                            logger.error(
                                f"[API ERROR] {uri} - Final failure: {final_error_msg}"
                            )
                            result_dict = {
                                "example_id": channel_id,
                                "audio_filepath": uri,
                                "transcript": None,
                                "error": final_error_msg,
                                "session_id": session_id,
                            }
                            results.append(result_dict)
                            async with checkpoint_write_lock:
                                with open(CHECKPOINT_FILE, "a") as f:
                                    f.write(json.dumps(result_dict) + "\n")

                        pbar.update(1)

                    if session_expired:
                        # Wipe completed records for this channel so we don't hit cache on next attempt
                        for entry in segment_entries:
                            completed_records.pop(entry["audio_filepath"], None)
                        # Rewind the progress bar for the cached hits we skipped
                        pbar.update(-len(results))
                        is_resumed = False
                        continue

                    try:
                        await upload_checkpoint_to_gcs()
                    except Exception as sync_err:
                        logger.warning(
                            f"Failed to backup checkpoint to GCS for channel {channel_id}: {sync_err}"
                        )

                    # Successfully completed! Break out of the attempt loop.
                    break

                finally:
                    # Only delete the session if we are NOT retrying, and the entire channel is successfully completed.
                    # If there are any failed/error segments, keep the session alive so we can resume it!
                    if session_id and not session_expired:
                        all_segments_succeeded = len(results) == len(
                            segment_entries
                        ) and all(
                            r.get("transcript") is not None for r in results
                        )
                        if all_segments_succeeded:
                            logger.info(
                                f"Purging completed session {session_id} for channel {channel_id}..."
                            )
                            for purge_retry in range(MAX_RETRIES):
                                try:
                                    async with session_quota_lock:
                                        await session_service.delete_session(
                                            session_id=session_id,
                                            user_id=channel_user_id,
                                            app_name=AGENT_ENGINE_ID,
                                        )
                                        await asyncio.sleep(
                                            0.2
                                        )  # Gentle 200ms purge stagger
                                    break
                                except Exception as purge_err:
                                    logger.warning(
                                        f"Purge attempt {purge_retry + 1} failed for session {session_id}: {purge_err}"
                                    )
                                    await asyncio.sleep(
                                        (2**purge_retry)
                                        + random.uniform(0.5, 1.5)
                                    )
    except FatalPipelineError:
        # Propagate fatal infrastructure/permission errors directly without swallowing them!
        raise
    except Exception as fatal_err:
        processed_filepaths = {r["audio_filepath"] for r in results}
        unprocessed_entries = [
            e
            for e in segment_entries
            if e["audio_filepath"] not in processed_filepaths
        ]

        logger.error(
            f"🚨 FATAL CHANNEL CRASH for {channel_id}: {fatal_err}. "
            f"Marking {len(unprocessed_entries)} remaining segments as failed and continuing other channels!"
        )

        for entry in unprocessed_entries:
            results.append(
                {
                    "example_id": channel_id,
                    "audio_filepath": entry["audio_filepath"],
                    "transcript": None,
                    "error": f"Fatal channel crash: {fatal_err}",
                    "session_id": session_id,
                }
            )
            pbar.update(1)

    return results


async def main() -> None:
    completed_records = {}
    channel_sessions = {}

    if OVERWRITE_EXISTING:
        logger.info("OVERWRITE_EXISTING is True. Wiping outputs and active cloud sessions.")
        
        # 1. Purge active sessions in the cloud before deleting the checkpoint files
        try:
            _, channel_sessions = load_gcs_checkpoint()
            if channel_sessions:
                session_service = VertexAiSessionService(
                    project=GCP_PROJECT_ID,
                    location=CONTROL_PLANE_LOCATION,
                )
                for channel_id, session_id in channel_sessions.items():
                    if session_id:
                        channel_user_id = f"{USER_ID}_{channel_id}"
                        logger.info(
                            f"Purging old cloud session {session_id} for channel {channel_id}..."
                        )
                        for purge_retry in range(MAX_RETRIES):
                            try:
                                async with session_quota_lock:
                                    await session_service.delete_session(
                                        session_id=session_id,
                                        user_id=channel_user_id,
                                        app_name=AGENT_ENGINE_ID,
                                    )
                                    await asyncio.sleep(0.2)
                                break
                            except Exception as purge_err:
                                logger.warning(
                                    f"Purge attempt {purge_retry + 1} failed for session {session_id}: {purge_err}"
                                )
                                await asyncio.sleep(
                                    (2**purge_retry) + random.uniform(0.5, 1.5)
                                )
        except Exception as checkpoint_err:
            logger.debug(
                f"No existing checkpoint found or failed to load. Skipping cloud session purge: {checkpoint_err}"
            )

        # 2. Delete local and remote output/checkpoint files
        out_bucket_name = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[
            0
        ]
        out_blob_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        out_blob = storage_client.bucket(out_bucket_name).blob(out_blob_path)
        if out_blob.exists():
            out_blob.delete()
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
        checkpoint_blob = get_gcs_checkpoint_blob()
        if checkpoint_blob.exists():
            checkpoint_blob.delete()
            logger.info("Deleted remote checkpoint file on GCS.")
    else:
        logger.info(
            "OVERWRITE_EXISTING is False. Resuming from GCS checkpoint..."
        )
        completed_records, channel_sessions = load_gcs_checkpoint()

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
    if not manifest_blob.exists():
        raise FileNotFoundError(f"Manifest not found at {MANIFEST_URI}")

    content = manifest_blob.download_as_text().strip().split("\n")
    channels = defaultdict(list)
    total_segments = 0

    # 🌿 100% FOOLPROOF MASTER GROUPING: Immutably groups all child slices under their standalone Parent Audio Recording unit folder name!
    for line in content:
        if line.strip():
            entry = json.loads(line)
            parent_audio_unit = Path(entry["audio_filepath"]).parent.name
            entry["example_id"] = parent_audio_unit
            channels[parent_audio_unit].append(entry)
            total_segments += 1

    # 🌿 Highly rigorous multi-dimensional chronological sort guaranteeing zero out-of-order segment splits!
    for ch in channels:

        def get_sort_key(x):
            return (
                x.get("offset", 0),
                x.get("start_time", 0),
                x.get("audio_filepath", ""),
            )

        channels[ch].sort(key=get_sort_key)

    active_channels = {}
    missing_total = 0
    for cid, entries in channels.items():
        missing = [
            e for e in entries if e["audio_filepath"] not in completed_records
        ]
        if missing:
            active_channels[cid] = entries
            missing_total += len(missing)

    if not active_channels:
        logger.info("Everything complete.")
    else:
        active_segments_count = sum(
            len(entries) for entries in active_channels.values()
        )
        logger.info(
            f"Resuming {len(active_channels)} active independent audio recording units. Processing {missing_total} pending segments."
        )

        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        with tqdm(
            total=active_segments_count, desc="Processing Transcriptions"
        ) as pbar:
            tasks = [
                asyncio.create_task(
                    process_single_channel(
                        cid,
                        entries,
                        completed_records,
                        channel_sessions,
                        semaphore,
                        pbar,
                    )
                )
                for cid, entries in active_channels.items()
            ]

            # Concurrent Fail-Fast Runner: If any task raises an exception, return immediately
            done, pending = await asyncio.wait(
                tasks, return_when=asyncio.FIRST_EXCEPTION
            )

            # Check if any task failed with an exception
            failure_exception = None
            for task in done:
                if task.exception():
                    failure_exception = task.exception()
                    break

            if failure_exception:
                logger.error(
                    f"🚨 FAIL-FAST TRIGGERED: A channel encountered a fatal error: {failure_exception}. "
                    "Cancelling all other active channels immediately!"
                )
                # Cancel all pending tasks
                for pending_task in pending:
                    pending_task.cancel()
                # Wait for pending tasks to be cancelled
                if pending:
                    await asyncio.gather(*pending, return_exceptions=True)
                # Propagate the fatal exception to halt notebook execution
                raise failure_exception

    # Final Summary and GCS Upload
    final_successes = {}
    final_errors = []

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            lines = f.readlines()

        final_ndjson = "".join(lines)
        out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
            final_ndjson
        )
        logger.info(f"Results saved to {CONSISTENT_OUTPUT_URI}")

        for line in lines:
            if line.strip():
                record = json.loads(line)
                if record.get("error"):
                    final_errors.append(record)
                else:
                    final_successes[record["audio_filepath"]] = record[
                        "transcript"
                    ]

    logger.info(
        f"Pipeline Complete. Total Successful Transcripts: {len(final_successes)}"
    )

    if final_errors:
        logger.error(
            f"⚠️ WARNING: {len(final_errors)} segments failed! Please re-run the pipeline cell to retry the failures."
        )
    else:
        logger.success("🎉 All segments processed successfully!")

    if final_successes:
        df = pd.DataFrame(
            list(final_successes.items()),
            columns=["audio_filepath", "transcript"],
        )
        df["example_id"] = df["audio_filepath"].apply(
            lambda x: Path(x).parent.name
        )
        display(df[["example_id", "audio_filepath", "transcript"]].head(10))

In [ ]:
# @title Execute Batch Transcriptions
await main()

In [ ]:
# @title 📊 Run Lookback Window Benchmarking Sweep (5 - 50)
# @markdown Run this cell to execute a high-fidelity, isolated, and unpolluted benchmarking sweep directly in your Cloud-Native Colab environment.
# @markdown This will evaluate ASR accuracy (WER/CER), latency, and Context Caching hit rates across lookback sizes.

import time
import json
import random
import math
from pathlib import Path
from collections import defaultdict
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Install jiwer for word error rate calculations
try:
    import jiwer
except ImportError:
    print("Installing jiwer...")
    !pip install -q jiwer
    import jiwer

# Simple text normalizer for ASR cleaning in Colab (numbers, punctuation, spacing)
def clean_text_for_wer(text: str) -> str:
    if not text:
        return ""
    text = text.lower().strip()
    text = re.sub(r'[^\w\s\-\:]', '', text) # Remove punctuation except hyphens/colons
    text = re.sub(r'\s+', ' ', text)       # Collapse spacing
    return text.strip()

async def run_lookback_benchmarking_sweep():
    # 1. Capture unique run timestamp to guarantee 100% session isolation
    run_timestamp = int(time.time())
    
    # 2. Load Manifest and Group by Channel
    print("📢 Loading manifest from GCS...")
    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
    
    if not manifest_blob.exists():
        raise FileNotFoundError(f"Manifest GCS file not found at {MANIFEST_URI}")
        
    content = manifest_blob.download_as_text().strip().split("
")
    channels = defaultdict(list)
    
    for line in content:
        if line.strip():
            entry = json.loads(line)
            parent_audio_unit = Path(entry["audio_filepath"]).parent.name
            entry["example_id"] = parent_audio_unit
            channels[parent_audio_unit].append(entry)
            
    # Chronological sort of segments
    for ch in channels:
        channels[ch].sort(key=lambda x: (x.get("offset", 0), x.get("start_time", 0), x.get("audio_filepath", "")))
        
    # Sweep configurations: Capped at 50 to focus on viable range and run 2x faster!
    context_sizes = [5, 10, 20, 30, 40, 50]
    
    # Calculate minimum segment depth needed (1 turn = 2 events)
    max_size = max(context_sizes)
    min_segments = int(max_size / 2)
    print(f"Filtering for channels with at least {min_segments} segments...")
    
    qualified_channels = {cid: entries for cid, entries in channels.items() if len(entries) >= min_segments}
    
    if not qualified_channels:
        raise RuntimeError(f"No channels found in the manifest with at least {min_segments} segments!")
        
    # Deterministically sample 3 channels for evaluation
    random.seed(42)
    sampled_cids = sorted(list(qualified_channels.keys()))
    sampled_cids = random.sample(sampled_cids, min(3, len(sampled_cids)))
    
    print(f"✅ Deterministically sampled {len(sampled_cids)} channels for evaluation (Seed: 42):")
    total_segments = 0
    for cid in sampled_cids:
        print(f"   - {cid}: {len(qualified_channels[cid])} segments")
        total_segments += len(qualified_channels[cid])
        
    raw_results = []
    
    # 3. Execute the Sweep
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    for size in context_sizes:
        print("\n" + "="*80)
        print(f"🚀 RUNNING SWEEP FOR CONTEXT WINDOW SIZE: {size} (NUM_RECENT_EVENTS)")
        print("="*80)
        
        # Build tasks to run sampled channels concurrently for this size
        async def evaluate_channel(cid, entries):
            results = []
            # Truly isolated user/session IDs incorporating run timestamp
            unique_user_id = f"eval_sweep_s{size}_{cid}_{run_timestamp}"
            unique_session_name = f"eval_sweep_{cid}_sz_{size}_{run_timestamp}"
            
            async with semaphore:
                # Create a fresh Vertex Session
                session_id = None
                for retry in range(5):
                    try:
                        async with session_quota_lock:
                            new_session = await session_service.create_session(
                                user_id=unique_user_id,
                                app_name=AGENT_ENGINE_ID,
                                display_name=unique_session_name,
                            )
                            await asyncio.sleep(0.5) # Quota stagger
                        if new_session and new_session.id:
                            session_id = new_session.id
                            break
                    except Exception as e:
                        print(f"⚠️ Failed to create session for {cid} (Attempt {retry+1}): {e}")
                        await asyncio.sleep((2**retry) + random.uniform(0.5, 1.5))
                        
                if not session_id:
                    print(f"❌ Failed to create session for {cid}. Skipping.")
                    return []
                    
                channel_runner = Runner(
                    agent=audio_agent,
                    app_name=AGENT_ENGINE_ID,
                    session_service=session_service,
                )
                
                run_config = RunConfig(
                    get_session_config=GetSessionConfig(
                        create_if_not_exists=False,
                        num_recent_events=size,
                    )
                )
                
                try:
                    # Sequentially process segments to build stateful history
                    for turn_index, entry in enumerate(entries, start=1):
                        uri = entry["audio_filepath"]
                        reference = entry.get("text") or entry.get("transcript") or entry.get("reference") or ""
                        
                        audio_part = types.Part.from_uri(file_uri=uri, mime_type="audio/flac")
                        message = types.Content(role="user", parts=[audio_part])
                        
                        transcript = None
                        latency = 0.0
                        prompt_tokens = 0
                        cached_tokens = 0
                        output_tokens = 0
                        is_fallback = False
                        err_msg = None
                        
                        req_start_time = time.time()
                        
                        # True Timeout Retries: wait_for is inside the retry decorator
                        @custom_async_retry
                        async def execute_stateful_turn_with_retry():
                            async def run_stream():
                                streamed_text = []
                                st_prompt = 0
                                st_cached = 0
                                st_output = 0
                                
                                async for event in channel_runner.run_async(
                                    user_id=unique_user_id,
                                    session_id=session_id,
                                    new_message=message,
                                    run_config=run_config,
                                ):
                                    if getattr(event, "content", None) and getattr(event.content, "parts", None):
                                        for part in event.content.parts:
                                            if hasattr(part, "text") and part.text:
                                                streamed_text.append(part.text)
                                                
                                    if event.is_final_response():
                                        usage = getattr(event, "usage_metadata", None)
                                        if usage:
                                            st_prompt = getattr(usage, "prompt_token_count", 0) or 0
                                            st_cached = getattr(usage, "cached_content_token_count", 0) or 0
                                            st_output = getattr(usage, "candidates_token_count", 0) or 0
                                            
                                return "".join(streamed_text).strip(), st_prompt, st_cached, st_output
                                
                            return await asyncio.wait_for(run_stream(), timeout=SOCKET_TIMEOUT)
                            
                        try:
                            res_text, p_tok, c_tok, o_tok = await execute_stateful_turn_with_retry()
                            transcript = res_text
                            prompt_tokens = p_tok
                            cached_tokens = c_tok
                            output_tokens = o_tok
                        except Exception as e:
                            print(f"❌ Stateful turn failed permanently for {Path(uri).name} under size {size}: {e}")
                            err_msg = str(e)
                            transcript = None
                            prompt_tokens = 0
                            cached_tokens = 0
                            output_tokens = 0
                            is_fallback = True
                                
                        latency = time.time() - req_start_time
                        
                        # Calculate WER/CER
                        ref_clean = clean_text_for_wer(reference)
                        hyp_clean = clean_text_for_wer(transcript)
                        
                        seg_wer = 1.0
                        seg_cer = 1.0
                        if ref_clean and transcript is not None:
                            try:
                                seg_wer = jiwer.wer(ref_clean, hyp_clean)
                                seg_cer = jiwer.cer(ref_clean, hyp_clean)
                            except Exception:
                                pass
                        elif not ref_clean and not hyp_clean and transcript is not None:
                            seg_wer = 0.0
                            seg_cer = 0.0
                            
                        results.append({
                            "context_size": size,
                            "channel_id": cid,
                            "audio_filepath": uri,
                            "turn_index": turn_index,
                            "reference": reference,
                            "hypothesis": transcript or "",
                            "latency": latency,
                            "prompt_tokens": prompt_tokens,
                            "cached_tokens": cached_tokens,
                            "output_tokens": output_tokens,
                            "is_fallback": is_fallback,
                            "wer": seg_wer,
                            "cer": seg_cer,
                            "error": err_msg
                        })
                finally:
                    # Purge session from Vertex AI database
                    if session_id:
                        try:
                            async with session_quota_lock:
                                await session_service.delete_session(
                                    session_id=session_id,
                                    user_id=unique_user_id,
                                    app_name=AGENT_ENGINE_ID,
                                )
                                await asyncio.sleep(0.2)
                        except Exception:
                            pass
            return results

        # Run channels concurrently for this size
        tasks = [evaluate_channel(cid, qualified_channels[cid]) for cid in sampled_cids]
        channel_results = await asyncio.gather(*tasks)
        
        # Flatten and accumulate
        size_results = [item for sublist in channel_results for item in sublist]
        raw_results.extend(size_results)
        
        # Compute quick aggregate metrics for display (exluding errors)
        df_size = pd.DataFrame(size_results)
        successful_df = df_size[df_size["error"].isna() | (df_size["error"] == "")]
        
        avg_wer = successful_df["wer"].mean() * 100 if not successful_df.empty else 0.0
        avg_cer = successful_df["cer"].mean() * 100 if not successful_df.empty else 0.0
        avg_lat = successful_df["latency"].mean() if not successful_df.empty else 0.0
        
        total_p = successful_df["prompt_tokens"].sum()
        total_c = successful_df["cached_tokens"].sum()
        c_rate = (total_c / total_p * 100) if total_p > 0 else 0
        failure_rate = (df_size["is_fallback"].sum() / len(df_size)) * 100
        
        print(f"📊 SUMMARY FOR SIZE {size}:")
        print(f"   - Average WER: {avg_wer:.2f}% | Average CER: {avg_cer:.2f}%")
        print(f"   - Average Latency: {avg_lat:.2f}s")
        print(f"   - Context Cache Hit Rate: {c_rate:.1f}% ({total_c:,} / {total_p:,} tokens)")
        print(f"   - Turn Failure Rate: {failure_rate:.1f}%")
        
    # 4. Save Raw Results
    df_raw = pd.DataFrame(raw_results)
    df_raw.to_csv("context_sweep_results.csv", index=False)
    print("\n💾 Raw results saved to 'context_sweep_results.csv'!")
    
    # 5. Generate Publication-Grade Statistical Report & Plots
    print("\n📈 Generating statistical charts...")
    
    summary_data = []
    for size in context_sizes:
        df_sub = df_raw[df_raw["context_size"] == size]
        successful_sub = df_sub[df_sub["error"].isna() | (df_sub["error"] == "")]
        n_segments = len(successful_sub)
        
        # WER Stats
        mean_wer = successful_sub["wer"].mean() * 100 if n_segments > 0 else 0.0
        sem_wer = (successful_sub["wer"].sem() * 100) if n_segments > 1 else 0.0
        ci_wer = 1.96 * sem_wer
        
        # CER Stats
        mean_cer = successful_sub["cer"].mean() * 100 if n_segments > 0 else 0.0
        sem_cer = (successful_sub["cer"].sem() * 100) if n_segments > 1 else 0.0
        ci_cer = 1.96 * sem_cer
        
        # Latency Stats
        mean_lat = successful_sub["latency"].mean() if n_segments > 0 else 0.0
        sem_lat = successful_sub["latency"].sem() if n_segments > 1 else 0.0
        ci_lat = 1.96 * sem_lat
        
        # Caching hit rate
        total_p = successful_sub["prompt_tokens"].sum()
        total_c = successful_sub["cached_tokens"].sum()
        cache_rate = (total_c / total_p * 100) if total_p > 0 else 0.0
        
        # Failure rate
        fail_rate = (df_sub["is_fallback"].sum() / len(df_sub)) * 100
        
        summary_data.append({
            "Context Size": size,
            "WER (%)": f"{mean_wer:.2f}% ± {ci_wer:.2f}%" if n_segments > 0 else "N/A",
            "CER (%)": f"{mean_cer:.2f}% ± {ci_cer:.2f}%" if n_segments > 0 else "N/A",
            "Avg Latency (s)": f"{mean_lat:.2f}s ± {ci_lat:.2f}s" if n_segments > 0 else "N/A",
            "Cache Hit Rate (%)": f"{cache_rate:.1f}%",
            "Failure Rate (%)": f"{fail_rate:.1f}%",
            "raw_wer_mean": mean_wer,
            "raw_wer_ci": ci_wer,
            "raw_cer_mean": mean_cer,
            "raw_cer_ci": ci_cer,
            "raw_lat_mean": mean_lat,
            "raw_lat_ci": ci_lat,
            "raw_cache_rate": cache_rate
        })
        
    df_sum = pd.DataFrame(summary_data)
    display(df_sum[["Context Size", "WER (%)", "CER (%)", "Avg Latency (s)", "Cache Hit Rate (%)", "Failure Rate (%)"]])
    
    # Render Plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.set_theme(style="whitegrid")
    
    sizes_numeric = [int(s) for s in context_sizes]
    
    # Plot 1: WER vs Context Size with CI bands
    axes[0].errorbar(
        sizes_numeric, df_sum["raw_wer_mean"], yerr=df_sum["raw_wer_ci"],
        fmt='-o', color='royalblue', ecolor='cornflowerblue', elinewidth=2, capsize=4, label="WER"
    )
    axes[0].set_title("Word Error Rate (WER) vs. Lookback Size\n(Lower is Better)", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Lookback Window (num_recent_events)", fontsize=10)
    axes[0].set_ylabel("WER (%)", fontsize=10)
    axes[0].set_xticks(sizes_numeric)
    
    # Plot 2: Cache Hit Rate vs Context Size
    axes[1].plot(sizes_numeric, df_sum["raw_cache_rate"], '-o', color='forestgreen', linewidth=2)
    axes[1].set_title("Context Cache Hit Rate vs. Lookback Size\n(Higher is Better)", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Lookback Window (num_recent_events)", fontsize=10)
    axes[1].set_ylabel("Cache Hit Rate (%)", fontsize=10)
    axes[1].set_xticks(sizes_numeric)
    axes[1].set_ylim(0, 105)
    
    # Plot 3: Latency vs Context Size with CI bands
    axes[2].errorbar(
        sizes_numeric, df_sum["raw_lat_mean"], yerr=df_sum["raw_lat_ci"],
        fmt='-o', color='darkorange', ecolor='moccasin', elinewidth=2, capsize=4, label="Latency"
    )
    axes[2].set_title("Average Turn Latency vs. Lookback Size\n(Lower is Better)", fontsize=12, fontweight='bold')
    axes[2].set_xlabel("Lookback Window (num_recent_events)", fontsize=10)
    axes[2].set_ylabel("Latency (seconds)", fontsize=10)
    axes[2].set_xticks(sizes_numeric)
    
    plt.suptitle(f"Gemini Stateful ASR Lookback Window Clean Evaluation (5 - 50)", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Run the sweep
await run_lookback_benchmarking_sweep()


In [ ]:
# @title Inspect Historical Session Content (Verify User & Model Event Retention)
# Inspect the event history of the active sessions to confirm audio and transcript retention
logger.debug("Fetching active sessions from Vertex AI...")
sessions_resp = await session_service.list_sessions(app_name=AGENT_ENGINE_ID)

if sessions_resp and sessions_resp.sessions:
    sample_session = sessions_resp.sessions[-1]  # Pick the latest session
    logger.debug(
        f"Retrieving session content for Session ID: {sample_session.id} (User: {sample_session.user_id})"
    )

    session_obj = await session_service.get_session(
        user_id=sample_session.user_id,
        app_name=AGENT_ENGINE_ID,
        session_id=sample_session.id,
    )

    if session_obj and hasattr(session_obj, "events") and session_obj.events:
        print(
            f"--- Active Session Events Summary ({len(session_obj.events)} total events) ---"
        )
        for idx, evt in enumerate(
            session_obj.events[:10]
        ):  # Inspect first 10 turns
            role = evt.author
            ts = evt.timestamp

            if role == "model":
                text_part = (
                    evt.content.parts[0].text
                    if evt.content and evt.content.parts
                    else "None"
                )
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Transcript: {text_part}"
                )
            elif role == "user":
                uri = "Audio Clip"
                if (
                    evt.content
                    and evt.content.parts
                    and hasattr(evt.content.parts[0], "file_data")
                ):
                    uri = evt.content.parts[0].file_data.file_uri
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Audio File: {uri}"
                )

        if len(session_obj.events) > 10:
            print(
                f"... and {len(session_obj.events) - 10} more events alternating between User (Audio) and Model (Transcript)."
            )
    else:
        print("No events recorded in this session yet.")
else:
    print(
        "No active sessions found. (Note: Sessions are automatically purged at the end of channel processing)."
    )

In [ ]:
# @title Validate Pipeline Integrity
# Count lines in manifest
m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
manifest_content = (
    storage_client.bucket(m_bucket)
    .blob(m_path)
    .download_as_text()
    .strip()
    .split("\n")
)
expected_count = len([l for l in manifest_content if l.strip()])

# Count lines in output
o_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
o_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
output_content = (
    storage_client.bucket(o_bucket)
    .blob(o_path)
    .download_as_text()
    .strip()
    .split("\n")
)
actual_count = len([l for l in output_content if l.strip()])

# Parse outputs to count failures
failures = []
for line in output_content:
    if line.strip():
        record = json.loads(line)
        if record.get("error"):
            failures.append(record)

print(f"--- Pipeline Validation ---")
print(f"Expected Segments (Manifest): {expected_count}")
print(f"Actual Transcripts (Output):   {actual_count}")
print(f"Failed Segments (Errors):      {len(failures)}")

if expected_count == actual_count:
    if failures:
        print(
            f"\n⚠️ WARNING: All segments were recorded, but {len(failures)} segments FAILED with errors!"
        )
        print("Please re-run the pipeline cell to retry the failures.")
    else:
        print(
            "\n✅ SUCCESS: All segments were transcribed successfully with zero errors!"
        )
else:
    print(
        f"\n❌ WARNING: Mismatch detected! Missing {expected_count - actual_count} segments."
    )

In [ ]:
# @title Session Isolation & Channel ID Audit


def audit_manifest_isolation():
    print(f"--- Auditing Manifest: {MANIFEST_URI} ---\n")

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    content = (
        storage_client.bucket(m_bucket)
        .blob(m_path)
        .download_as_text()
        .strip()
        .split("\n")
    )

    channel_to_paths = defaultdict(set)
    path_to_channels = defaultdict(set)

    for line in content:
        if not line.strip():
            continue
        entry = json.loads(line)
        cid = entry.get("example_id")
        path = entry.get("audio_filepath")

        # Extract the source folder name from the path as a 'ground truth' source
        source_folder = path.split("/")[-2] if "/" in path else "unknown"

        channel_to_paths[cid].add(source_folder)
        path_to_channels[source_folder].add(cid)

    # Check 1: Does one channel ID map to multiple physical folders? (Session Hijacking)
    overlap_found = False
    for cid, sources in channel_to_paths.items():
        if len(sources) > 1:
            print(
                f"⚠️ COLLISION: Channel ID '{cid}' is being shared by multiple sources: {sources}"
            )
            print("   This WILL cause session hijacking and context pollution.")
            overlap_found = True

    # Check 2: Does one physical folder have multiple channel IDs?
    for source, cids in path_to_channels.items():
        if len(cids) > 1:
            print(
                f"ℹ️ Note: Source '{source}' is split across multiple IDs: {cids}"
            )

    if not overlap_found:
        print(
            "✅ Isolation Audit Passed: Each Channel ID maps to exactly one source directory."
        )

    print(f"\nUnique Sessions to be created: {len(channel_to_paths)}")


audit_manifest_isolation()